# PEST_HP con el modelo Freyberg: guía práctica paso a paso

Este notebook es el ejemplo ejecutable que acompaña a la guía [`docs/guia_pest_hp.md`](../../docs/guia_pest_hp.md). Usa **el mismo problema que los notebooks de la parte 1**: el modelo Freyberg (MODFLOW 6 + MODPATH 7) con pilot points para K y recarga (como en `part1_08_pilotpoints_run` y `part1_09_regularization`), pero lo calibra con **PEST_HP** en vez de PEST++GLM.

PEST_HP es la versión paralela de PEST para "alto rendimiento" (de John Doherty, Watermark Numerical Computing). Usa el mismo método de Gauss-Levenberg-Marquardt (GLM) que PEST y PEST++GLM, el mismo archivo de control (versión 1), los mismos archivos de plantilla (`.tpl`) e instrucciones (`.ins`), y reparte las corridas del modelo entre un **manager** (`pest_hp`) y varios **agents** (`agent_hp`).

Qué cambia respecto a PEST++ (y lo que hace este notebook):

| Tema | PEST++ (resto del repo) | PEST_HP (este notebook) |
|---|---|---|
| Nombres de parámetros | sin límite | máximo 12 caracteres → `hp.shorten_par_names()` |
| Opciones `++` | se leen | no se usan → `hp.prepare_pst_for_pest_hp()` |
| Versión del archivo de control | 1 o 2 | solo versión 1 → `hp.write_pst_for_pest_hp()` |
| Ejecución en paralelo | `pyemu.os_utils.start_workers()` | `pest_hp` + `agent_hp` → `hp.start_pest_hp()` |
| Historia de phi | `case.iobj` (csv) | `case.ofr` (tabla) → `hp.read_ofr()` |
| Incertidumbre posterior | `case.par.usum.csv`, `case.pred.usum.csv` | `case.jco` + `pyemu.Schur` (FOSM) |

### Requisitos
1. El entorno del repo funcionando: `pixi install` y `pixi run get-exes` (ver README). En Windows, `get-exes` también descarga e instala la suite PEST_HP (`pest_hp.exe`, `agent_hp.exe`, ...) en el entorno pixi. Si solo falta PEST_HP: `pixi run get-pest-hp`.
2. `get-exes` también instala `pestchek`, `tempchek` e `inschek` (de la suite de PEST) para verificar el archivo de control, las plantillas y las instrucciones.
3. PEST_HP rinde mejor con 10 agentes o más (manual de PEST_HP, sección 1.8); con menos funciona, pero más lento.

In [ ]:
import sys
import os
import shutil
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psutil

import pyemu
import flopy

# repo helpers used by all the part1 notebooks
sys.path.insert(0, "..")
import herebedragons as hbd
# PEST_HP helpers (same folder as this notebook)
import pest_hp_helpers as hp

plt.rcParams['font.size'] = 10
pyemu.plot_utils.font = 10

## 1. ¿Dónde están los ejecutables de PEST_HP?

Si instaló PEST_HP con `pixi run get-exes` (o `pixi run get-pest-hp`), deje `PEST_HP_DIR = None`: los ejecutables ya están en el PATH del entorno. Si lo tiene en otra carpeta (por ejemplo `C:\PEST_HP`), escriba esa ruta.

Si PEST_HP no se encuentra, el notebook igual prepara todos los archivos y solo se saltan las celdas que corren PEST_HP (`RUN_PEST_HP = False`). Así puede revisar el archivo de control antes de instalar nada.

In [ ]:
# folder with pest_hp(.exe) and agent_hp(.exe); None = look on the PATH (pixi environment)
PEST_HP_DIR = None  # e.g. r"C:\PEST_HP"

pest_hp_path = hp.find_exe("pest_hp", PEST_HP_DIR)
agent_hp_path = hp.find_exe("agent_hp", PEST_HP_DIR)
print("pest_hp :", pest_hp_path)
print("agent_hp:", agent_hp_path)

# only run PEST_HP if both executables are available
RUN_PEST_HP = pest_hp_path is not None and agent_hp_path is not None
if not RUN_PEST_HP:
    print("\nPEST_HP not found: the files will be prepared but PEST_HP will not be run")

## 2. Construir la interfaz de PEST (igual que en la parte 1)

Usamos exactamente las mismas funciones que `part1_08_pilotpoints_run`:
- `hbd.prep_pest()` crea el archivo de control base (`freyberg.pst`), las plantillas y las instrucciones.
- `hbd.add_ppoints()` agrega pilot points para K (`hk1`) y para la recarga (`rchpp`), los pozos (`wel`), el caudal de entrada del río (`strinf`) y escribe `forward_run.py`, que interpola los pilot points a la grilla y corre MODFLOW 6 y MODPATH 7.

> En su propio modelo, este paso es su propia interfaz: archivos `.tpl`/`.ins` hechos a mano, con `pyemu.Pst.from_io_files()`, o con `pyemu.utils.PstFrom(..., longnames=False)` (ver `part2_01_pstfrom_pest_setup`). Lo que sigue funciona igual en cualquier caso.

In [ ]:
# folder containing the original model files
org_d = os.path.join('..', '..', 'models', 'monthly_model_files_1lyr_newstress')
# a folder to hold a copy of the original model files (our "template")
tmp_d = os.path.join('freyberg_mf6')
if os.path.exists(tmp_d):
    shutil.rmtree(tmp_d)
shutil.copytree(org_d, tmp_d)
# copy mf6, mp7 (and the PEST++ executables) into the template folder
hbd.prep_bins(tmp_d)
# base PEST interface and the pilot point version of it
hbd.prep_pest(tmp_d)
hbd.add_ppoints(tmp_d)

In [ ]:
pst = pyemu.Pst(os.path.join(tmp_d, 'freyberg_pp.pst'))
print("parameters:", pst.npar, " adjustable:", pst.npar_adj)
print("observations:", pst.nobs, " non-zero weight:", pst.nnz_obs)
pst.parameter_data.head()

## 3. Hacer el archivo de control compatible con PEST_HP

### 3.1 Diagnóstico
`hp.check_pest_hp_compat()` revisa los límites de nombres de PEST (parámetros ≤ 12 caracteres; observaciones e información previa ≤ 20; grupos ≤ 12), los espacios y las opciones `++` de PEST++.

In [ ]:
problems = hp.check_pest_hp_compat(pst)

Los nombres de los pilot points que genera pyEMU (`hk_i:12_j:12_zone:1`) son demasiado largos para PEST_HP.

### 3.2 Acortar los nombres de los parámetros
`hp.shorten_par_names()` renombra cada parámetro largo como `<grupo>_<número>` (`hk1_0001`, `rchpp_0001`, ...) **en el archivo de control y en todas las plantillas `.tpl`**, y guarda la tabla de equivalencias en `par_name_map.csv`.

> Hágalo **antes** de agregar información previa (regularización): las ecuaciones de información previa usan los nombres de los parámetros.

In [ ]:
name_map = hp.shorten_par_names(pst, tmp_d)
pd.read_csv(os.path.join(tmp_d, 'par_name_map.csv')).head()

### 3.3 Quitar las opciones de PEST++ y escribir un archivo de control versión 1
Los pronósticos (`forecasts`) eran una opción `++` de PEST++. Para PEST_HP siguen siendo observaciones con peso cero: quedan en la matriz Jacobiana (`.jco`) y más adelante los usamos para el análisis FOSM con pyEMU.

In [ ]:
# keep the forecast names: they are zero-weight observations
forecasts = pst.pestpp_options['forecasts'].split(',')
print(forecasts)
hp.prepare_pst_for_pest_hp(pst)

# a test run: noptmax=0 runs the model once with the initial parameter values
pst.control_data.noptmax = 0
case = 'freyberg_hp'
hp.write_pst_for_pest_hp(pst, os.path.join(tmp_d, case + '.pst'))
problems = hp.check_pest_hp_compat(pst)
assert len(problems) == 0

### 3.4 Copiar PEST_HP en la carpeta plantilla y correr `pestchek`
Igual que `hbd.prep_bins()` copia MODFLOW y PEST++, `hp.prep_pest_hp_bins()` copia `pest_hp`, `agent_hp` (y `pestchek`, `tempchek`, `inschek` si los encuentra) en la carpeta plantilla, para que cada carpeta de agente sea completa.

`pestchek` es la verificación oficial del archivo de control: **corríjalo hasta que no muestre errores** antes de lanzar PEST_HP.

In [ ]:
if RUN_PEST_HP:
    hp.prep_pest_hp_bins(tmp_d, exe_dir=PEST_HP_DIR)
    hp.run_pestchek(case, cwd=tmp_d, exe_dir=PEST_HP_DIR)

## 4. Corrida de prueba (`NOPTMAX = 0`)

Siempre empiece así: PEST_HP corre el modelo una sola vez y calcula phi. Si esto falla, el problema está en la interfaz (plantillas, instrucciones, comando del modelo) y no en la calibración.

Cómo se lanza PEST_HP (lo que hace `hp.start_pest_hp()` por usted):

```
manager (carpeta master):   pest_hp freyberg_hp /h :4004
agente i (carpeta worker_i): agent_hp freyberg_hp /h localhost:4004
```

En Windows puede aparecer el aviso del firewall la primera vez: permita el acceso (o cancele: en la misma máquina con `localhost` igual funciona).

In [ ]:
m_d = 'master_hp'
if RUN_PEST_HP:
    hp.start_pest_hp(tmp_d, case + '.pst', num_agents=1, master_dir=m_d,
                     pest_hp_exe=pest_hp_path, agent_exe=agent_hp_path)
    pst_test, _, _ = hp.load_results(m_d, case + '.pst')
    print("phi with the initial parameters:", pst_test.phi)
    print(pst_test.phi_components)

## 5. Calibración en modo `estimation`

### 5.1 Variables de control más útiles en PEST_HP
Están en la sección `* control data` y pyEMU las expone en `pst.control_data`:

| Variable | Qué hace | Valor sugerido |
|---|---|---|
| `noptmax` | número máximo de iteraciones | 8–30 |
| `derforgive` | si una corrida falla al calcular derivadas, PEST_HP sigue en lugar de detenerse | `derforgive` |
| `jacupdate` | actualizaciones de Broyden de la Jacobiana tras las pruebas de lambda, seguidas de una segunda tanda de pruebas | 999 (todas); 0 = ninguna, suele convenir con muchos agentes |
| `relparmax`, `facparmax` | límite de cambio de los parámetros por iteración | 10 |
| `phiredstp`, `nphistp`, `nphinored`, `relparstp`, `nrelpar` | criterios de parada | por defecto |
| `rstfle` | `restart` permite reanudar con `/s` | `restart` |

PEST_HP **ignora** `rlambda1`, `rlamfac`, `phiratsuf`, `phiredlam` y `numlam`: elige él mismo las lambdas de Marquardt (nunca menos de tres, según el nº de agentes). También activa siempre `lamforgive` internamente. Esas variables se dejan con los valores de pyEMU para que `pestchek` no se queje. Las variables propias de PEST_HP (`UPTESTMIN`, `UPTESTLIM`, ...) se ven en la guía, sección 6.1.

In [ ]:
pst.control_data.pestmode = 'estimation'
pst.control_data.noptmax = 8
# tolerate failed model runs while filling the Jacobian (failed upgrade tests are always tolerated by PEST_HP)
pst.control_data.derforgive = 'derforgive'
# Broyden update of the Jacobian using the upgrade-testing runs
pst.control_data.jacupdate = 999
hp.write_pst_for_pest_hp(pst, os.path.join(tmp_d, case + '.pst'))

In [ ]:
# one agent per physical core is a good start (PEST_HP works best with 10 or more)
num_agents = psutil.cpu_count(logical=False)
num_agents

In [ ]:
if RUN_PEST_HP:
    hp.start_pest_hp(tmp_d, case + '.pst', num_agents=num_agents, master_dir=m_d,
                     pest_hp_exe=pest_hp_path, agent_exe=agent_hp_path)

### 5.2 Revisar los resultados
Archivos principales de PEST_HP en la carpeta del manager:

| Archivo | Contenido |
|---|---|
| `case.rec` | registro completo de la corrida (phi por iteración, lambdas, parámetros, estadísticos) |
| `case.par` | valores óptimos de los parámetros |
| `case.rei` | residuos de la mejor corrida (pyEMU lo lee directamente) |
| `case.jco` | matriz Jacobiana (binaria; `pyemu.Jco.from_binary`) |
| `case.sen`, `case.seo` | sensibilidades compuestas de parámetros y observaciones |
| `case.ofr` | tabla de phi (total y por grupo) por iteración, propia de PEST_HP → `hp.read_ofr()` |
| `case.rmr` | registro de la gestión de corridas en paralelo (útil si un agente falla) |
| `case.rme` | eficiencia de cada agente: corridas, fallas, tiempos (propio de PEST_HP) |

In [ ]:
def phi_history(m_d, case):
    # PEST_HP's objective function record file; fall back to the run record
    df = hp.read_ofr(os.path.join(m_d, case + '.ofr'))
    if df.empty:
        df = hp.read_rec_phi(os.path.join(m_d, case + '.rec'))
    if not df.empty:
        ax = df.plot(marker='o', logy=True, figsize=(8, 4))
        ax.set_ylabel('phi')
    return df

if RUN_PEST_HP:
    pst_est, par_est, jco_est = hp.load_results(m_d, case + '.pst')
    print("final phi:", pst_est.phi)
    df_phi = phi_history(m_d, case)
    hbd.plot_1to1(pst_est, title=f'PEST_HP estimation - total phi: {round(pst_est.phi, 2)}')

Mapa de K calibrada: se cargan los parámetros óptimos, se escriben los archivos de entrada y se vuelve a correr el modelo en la carpeta del manager (igual que en `part1_08`).

In [ ]:
def plot_k(m_d, case, title):
    # load the control file and replace the initial values with the estimated ones
    p = pyemu.Pst(os.path.join(m_d, case + '.pst'))
    p.parrep(os.path.join(m_d, case + '.par'))
    p.write_input_files(pst_path=m_d)
    # forward_run.py interpolates the pilot points to the grid and runs the model
    pyemu.os_utils.run('python forward_run.py', cwd=m_d)
    sim = flopy.mf6.MFSimulation.load(sim_ws=m_d, verbosity_level=0)
    gwf = sim.get_model()
    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(1, 1, 1, aspect='equal')
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    ca = mm.plot_array(np.log10(gwf.npf.k.get_data()), masked_values=[1e30])
    cb = plt.colorbar(ca, shrink=0.5)
    cb.ax.set_title('$Log_{10}K$')
    # pilot point locations
    df_pp = pyemu.pp_utils.pp_tpl_to_dataframe(os.path.join(m_d, 'hkpp.dat.tpl'))
    ax.scatter(df_pp.x, df_pp.y, marker='x', c='k', s=15)
    mm.plot_inactive()
    ax.set_title(title)
    return gwf

if RUN_PEST_HP:
    plot_k(m_d, case, 'K - PEST_HP estimation')
    hbd.plot_truth_k(m_d);

## 6. Calibración regularizada (modo `regularization`)

Con muchos parámetros (aquí 65) el problema inverso está mal planteado. Igual que en `part1_09_regularization`, agregamos regularización de Tikhonov:
- **Valor preferido** (orden cero): cada parámetro "prefiere" su valor inicial.
- **Diferencia preferida** (primer orden): pilot points cercanos prefieren valores parecidos, con pesos derivados de un variograma.

PEST_HP ajusta el peso de la regularización para que la phi de medición llegue a `PHIMLIM`. Recuerde: **los grupos de información previa deben empezar con `regul`** (pyEMU ya los nombra así).

In [ ]:
pst_reg = pyemu.Pst(os.path.join(tmp_d, case + '.pst'))
# preferred value regularization, weights from the parameter bounds
pyemu.helpers.zero_order_tikhonov(pst_reg, parbounds=True)
# preferred difference regularization between pilot points, weights from a variogram
v = pyemu.geostats.ExpVario(contribution=1.0, a=2500.0)
gs = pyemu.geostats.GeoStruct(variograms=v, nugget=0.0)
# the template file already holds the SHORT parameter names
df_pp = pyemu.pp_utils.pp_tpl_to_dataframe(os.path.join(tmp_d, 'hkpp.dat.tpl'))
cov = gs.covariance_matrix(df_pp.x, df_pp.y, df_pp.parnme)
pyemu.helpers.first_order_pearson_tikhonov(pst_reg, cov, reset=False)
pst_reg.prior_information.obgnme.unique()

In [ ]:
pst_reg.control_data.pestmode = 'regularization'
# target measurement phi: about the number of non-zero weighted observations
pst_reg.reg_data.phimlim = pst_reg.nnz_obs
pst_reg.reg_data.phimaccept = 1.1 * pst_reg.reg_data.phimlim
pst_reg.control_data.noptmax = 10
case_reg = 'freyberg_hp_reg'
hp.write_pst_for_pest_hp(pst_reg, os.path.join(tmp_d, case_reg + '.pst'))
hp.check_pest_hp_compat(pst_reg)
if RUN_PEST_HP:
    hp.run_pestchek(case_reg, cwd=tmp_d, exe_dir=PEST_HP_DIR)

In [ ]:
m_d_reg = 'master_hp_reg'
if RUN_PEST_HP:
    hp.start_pest_hp(tmp_d, case_reg + '.pst', num_agents=num_agents, master_dir=m_d_reg,
                     pest_hp_exe=pest_hp_path, agent_exe=agent_hp_path)
    pst_r, par_r, jco_r = hp.load_results(m_d_reg, case_reg + '.pst')
    print("measurement phi:", sum(v for k, v in pst_r.phi_components.items() if not k.startswith('regul')))
    print("phimlim:", pst_r.reg_data.phimlim)
    df_phi = phi_history(m_d_reg, case_reg)
    hbd.plot_1to1(pst_r, title='PEST_HP regularization')
    plot_k(m_d_reg, case_reg, 'K - PEST_HP regularization')

## 7. Post-proceso con pyEMU a partir de la Jacobiana

PEST_HP no escribe los archivos `*.usum.csv` de PEST++, pero sí deja la Jacobiana (`case.jco`). Con ella, pyEMU hace el mismo análisis lineal que los notebooks `part1_10_intro_to_fosm` y `part1_11_local_and_global_sensitivity`:
- **Identificabilidad** de los parámetros (`pyemu.ErrVar`).
- **Incertidumbre de los pronósticos** antes y después de calibrar (FOSM, `pyemu.Schur`), con la incertidumbre previa tomada de los límites de los parámetros.

> Use la Jacobiana de la corrida en modo `estimation`: ahí la información previa no actúa como datos.

In [ ]:
if RUN_PEST_HP:
    jco_file = os.path.join(m_d, case + '.jco')
    pst_fosm = pyemu.Pst(os.path.join(m_d, case + '.pst'))
    # identifiability with the first 20 singular values
    ev = pyemu.ErrVar(jco=jco_file, pst=pst_fosm)
    ident = ev.get_identifiability_dataframe(singular_value=20)
    ax = ident.ident.sort_values(ascending=False).iloc[:30].plot(kind='bar', figsize=(12, 4))
    ax.set_ylabel('identifiability')

In [ ]:
if RUN_PEST_HP:
    # prior parameter covariance from the bounds, observation noise from the weights
    sc = pyemu.Schur(jco=jco_file, pst=pst_fosm, forecasts=forecasts)
    df_fore = sc.get_forecast_summary()
    display(df_fore)
    # how much each parameter group contributes to forecast uncertainty
    display(sc.get_par_group_contribution())

## 8. Detener y reanudar una corrida

- **Detener:** `Ctrl+C` en la ventana de PEST_HP, o `pstop` / `pstopst` desde otra terminal abierta en la carpeta del manager (`pstopst` escribe antes el estado en el `.rec`).
- **Reanudar:** con `RSTFLE = restart` en el archivo de control, una corrida detenida o cortada (apagón, Windows Update, etc.) se retoma en la misma carpeta del manager con el switch **`/s`**, que va antes de `/h`:

```
pest_hp freyberg_hp /s /h :4004
```

Si se cortó mientras llenaba la Jacobiana, PEST_HP solo hace las corridas que faltaban. Los agentes se relanzan con el comando de siempre (sin `/s`). Con el helper:

In [ ]:
# hp.start_pest_hp(tmp_d, case + '.pst', num_agents=num_agents, master_dir=m_d,
#                  pest_hp_exe=pest_hp_path, agent_exe=agent_hp_path, restart_flag='/s')

## 9. Llevar esto a su propio modelo

1. Arme o reutilice su interfaz PEST (plantillas, instrucciones, `forward_run.py` o `.bat`). Si usa `PstFrom`, ponga `longnames=False`.
2. `hp.check_pest_hp_compat(pst)` → `hp.shorten_par_names(pst, template_dir)` si hace falta → `hp.prepare_pst_for_pest_hp(pst)`.
3. Escriba el `.pst` con `hp.write_pst_for_pest_hp()` y corra `pestchek` (y `tempchek`/`inschek` si tiene dudas con una plantilla o instrucción).
4. Corrida de prueba con `NOPTMAX = 0` y 1 agente.
5. Calibración con `hp.start_pest_hp(...)`: primero `estimation` con pocos parámetros, después `regularization`.
6. Post-proceso: `case.ofr`, `case.rec`, `case.rei`, `case.par`, y `case.jco` con pyEMU.

La guía completa (instalación, comandos en varias máquinas, solución de problemas) está en [`docs/guia_pest_hp.md`](../../docs/guia_pest_hp.md).